In [7]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
import os
url = "https://365datascience.com/courses/"

# Step 1: Load raw documents from the website
loader = WebBaseLoader(url)
raw_documents = loader.load()  # <-- this was missing

# Step 2: Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

# Step 3: Create embeddings
embeddings = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))

# Step 4: Store in FAISS
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# Step 5: LLM for generating answers
llm = ChatOpenAI(
    openai_api_key=config.api_key,
    model="gpt-4.1-mini",
    temperature=0
)

# Step 6: Manual memory
chat_history = []

# Step 7: User query
query = "Which course on 365DataScience can help me learn AI?"
relevant_docs = retriever.invoke(query)

# Step 8: Build context
context = "\n\n".join(doc.page_content for doc in relevant_docs)

# Step 9: Build conversation history
history_text = "\n".join(
    f"User: {q}\nAssistant: {a}" for q, a in chat_history
)

# Step 10: Build prompt
prompt = f"""
Use the context below to answer the question.

Conversation history:
{history_text}

Context:
{context}

Question:
{query}
"""

# Step 11: Generate response
response = llm.invoke(prompt)
chat_history.append((query, response.content))

print(response.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}